# Reading the Inference Report

This example is part of the *Universal Inference* section. With `fallback = LaplaceApproximation()`, RxInfer runs exact rules wherever a model has them, and approximates everywhere else. One question is left that the posteriors alone cannot answer: **what was approximated, and how?**

`infer(...; report = true)` answers it. The result gets a report, `result.report`, with two sections:

- `result.report.approximations` lists each approximated posterior and how it was projected. It gives the node instances whose messages formed it, by an exact rule or by the rule fallback, and where the fallback's expectations went. It also warns about the parts of the inference most likely to be inaccurate.
- `result.report.timings` says where the time went.

This notebook reads the report of one model, under two inference schemes, and then uses it as data.

In [ ]:
using RxInfer, StatsFuns, StableRNGs, PrettyTables

## The model

A logistic regression with one coefficient $\beta$, whose prior precision $\tau$ is itself unknown:

$$\begin{aligned}
\tau &\sim \mathrm{Gamma}(2, 1),\\
\beta &\sim \mathcal{N}(0, \tau^{-1}),\\
y_i &\sim \mathrm{Bernoulli}\left(\sigma(\beta x_i)\right).
\end{aligned}$$

The Gamma prior on a Gaussian's precision is conjugate under mean field, so RxInfer has exact variational rules for $\tau$. The likelihood has none: we write it as a node of our own, with a log-density and no rules.

In [ ]:
rng = StableRNG(3)
n = 60
x = randn(rng, n)
β_true = 1.5
y = [rand(rng) < logistic(β_true * xi) ? 1.0 : 0.0 for xi in x]

struct LogisticBernoulli{B, X} <: DiscreteUnivariateDistribution
    β::B
    x::X
end

BayesBase.logpdf(d::LogisticBernoulli, y) = y * log(logistic(d.β * d.x)) + (1 - y) * log1p(-logistic(d.β * d.x))
BayesBase.insupport(::LogisticBernoulli, y) = true

@define_factor_node(node = LogisticBernoulli, type = Stochastic, interfaces = [:out, :β, :x])

@model function shrinkage(y, x)
    τ ~ Gamma(shape = 2.0, rate = 1.0)
    β ~ NormalMeanPrecision(0.0, τ)
    for i in eachindex(y)
        y[i] ~ LogisticBernoulli(β, x[i])
    end
end

## Under mean field

We factorise the posterior as $q(\beta)\,q(\tau)$ and ask for the report.

In [ ]:
mean_field = @constraints begin
    q(β, τ) = q(β)q(τ)
end

result = infer(
    model = shrinkage(x = x),
    data = (y = y,),
    constraints = mean_field,
    iterations = 10,
    fallback = LaplaceApproximation(),
    free_energy = true,
    report = true,
)

q_β, q_τ = result.posteriors[:β][end], result.posteriors[:τ][end]
@assert abs(mean(q_β) - β_true) < 2 * std(q_β) # hide

With PrettyTables.jl loaded, the report prints as tables.

In [ ]:
approximations = result.report.approximations # hide
@assert isempty(RxInfer.diagnostics(result.report)) # hide
@assert RxInfer.is_approximated(approximations.variables[(:β, nothing)]) # hide
@assert !RxInfer.is_approximated(approximations.variables[(:τ, nothing)]) # hide
result.report.approximations

How to read it:

- **The summary line.** One of the two posteriors was approximated. The other, $\tau$'s, is exact: its Gamma posterior comes from the conjugate rules. Nothing calls for a warning.
- **Approximated posteriors.** Each approximated posterior is a group of rows. Its header says how it was projected: here by Laplace's method, in how many Newton iterations, and with which log-normaliser. Each row is a node instance as the model writes it, and instances of one shape are folded into one row: `y[1:60] ~ LogisticBernoulli(β, …)`, the `…` standing for the inputs $x_i$, which differ.
- **The counts.** The columns say how many of a node's messages towards the variable came from an exact rule and how many from the rule fallback. They cover all ten iterations. The fallback's expectations are exact: under mean field, the likelihood's other inputs are the data and the constants $x_i$.
- **Joints and energies by the fallback.** These are the likelihood's average energies, which the free energy needs.
- **Expectations.** The expectation backends the session had, in the order they are tried.

## Under belief propagation, a warning

Without the factorisation, inference is belief propagation. The Gaussian node $\beta \sim \mathcal{N}(0, \tau^{-1})$ then has no exact rule, because its precision is random. The fallback takes over there too, and the report flags it.

In [ ]:
result_bp = infer(
    model = shrinkage(x = x),
    data = (y = y,),
    fallback = LaplaceApproximation(),
    report = true,
)
found = RxInfer.diagnostics(result_bp.report)
@assert [d.kind for d in found] == [:expectation_over_messages] # hide
result_bp.report.approximations

The warning, `:expectation_over_messages`, says what happened. Under belief propagation, the fallback's message towards $\beta$ is an expectation over the *message* of $\tau$. That expectation is the variational message, not the belief propagation message, which would integrate the node against $\tau$'s message instead. The result is an approximation of belief propagation, and here an uncontrolled one. Two cures:

- **Mean field.** With the factorisation above, the expectation over a marginal is exactly what variational message passing prescribes, and the warning goes away.
- **Cavity methods.** Expectation propagation computes such messages properly.

Each diagnostic is plain data, with a kind, a level, a subject and a sentence:

In [ ]:
found

A test, or a notebook such as this one, can assert that an inference raised no warnings: `isempty(filter(d -> d.level === :warning, RxInfer.diagnostics(result.report)))`.

## The report as data

Everything the display shows is plain data, which the display only formats:
- `RxInfer.node_types(report)` counts the computations by node type;
- `report.variables` records, for each variable, its projections and the messages that formed it;
- `report_tables` gives the tables themselves.

For example, the number of times the fallback computed a message of the likelihood towards $\beta$:

In [ ]:
RxInfer.node_types(result.report.approximations)[(LogisticBernoulli, :message, :β)].fallbacks

And $\beta$'s last Laplace projection, as `LaplaceForm` reported it:

In [ ]:
result.report.approximations.variables[(:β, nothing)].marginal.outcome

The rows of the posteriors' table:

In [ ]:
first(report_tables(result.report.approximations)).rows

## Timings

The second section of the report says where the time went. Julia compiles a model the first time it runs, so we look at a second run.

In [ ]:
infer(model = shrinkage(x = x), data = (y = y,), constraints = mean_field, iterations = 10, fallback = LaplaceApproximation(), free_energy = true) # hide
timed = infer(
    model = shrinkage(x = x),
    data = (y = y,),
    constraints = mean_field,
    iterations = 10,
    fallback = LaplaceApproximation(),
    free_energy = true,
    report = (:timings,),
)
@assert length(timed.report.timings.iterations) == 10 # hide
timed.report

Each call's own time is counted: the time of what it called goes to its own kind, so the kinds add up. What no rule, fallback or projection took is the engine's own time: the products of messages, the streams between the nodes and the entropies of the free energy. In this small model, most of the time is the engine's. Of the approximations, the Laplace projections of $\beta$ and the fallback's quadrature over the 60 observations take a few percent each, and the exact rules almost nothing. For statistics over many runs, RxInfer's benchmark callbacks are the tool.

## Summary

- `report = true` records what an inference approximated, by posterior, by node instance and by rule, at no cost when it is off.
- `RxInfer.diagnostics` names the parts most likely to be inaccurate. Here, a fallback averaging over a message under belief propagation, which mean field cures.
- The report is data: its counts, projections and tables can be read, and checked, directly.